# Parameter estimation and fitting: Extra exercise
### The $H \to \gamma \gamma$ Fit
This is an example of fitting the 2-photon invariant mass to determine the number of Higgs signal events.

We have a data set consists of ~ 30000 di-photon invariant mass values from a tetx file [Hgg.txt](Hgg.txt)  which should be in the current directory. We perform a binned maximum likelihood fit (for reducing the CPU time) using RooFit. 
[RooFit](https://root.cern.ch/roofit-20-minutes) is a toolkit for data modeling which allows to modeling probability distributions within ROOT in a compact and abstract way.

## Startup ROOT
Import the ROOT module: this will activate the integration layer with the notebook automatically

In [1]:
import ROOT
import numpy as np
print(ROOT.gROOT.GetVersion())

6.38.00


## 1. Reading Input data set
The ROOT `TTree.ReadFile` is able to read ASCII files and re-interpret their content as the user decide (csv, tsv, ...).

The data were are feeding is expressing the measured invariant mass of a $\gamma$-$\gamma$ system measured at each event. The data is obtained from CMS during the Run-I period (2010-2012)

In [2]:
# Create a TTree to organize the data: call it "tree"
tree = ROOT.TTree("tree","tree with data from H to gg simulation")

# Read the ASCII file using the tree: the column find in the file is filling the branch 'x"
nevt = tree.ReadFile("Hgg.txt","x")
if nevt <= 0:
    raise RuntimeError("Error reading data from input file ")

# Print out how many events did you read
print(f"Number of events read: {nevt}")

Number of events read: 30770


## 2.  Create a histogram representing the  data

We make an histogram with 100 bins from 110 to 160 where we fill the invariant mass data from the tree

In [3]:
# Instantiate a TH1D with 100 bins defined between 110 and 160
histo_title="Invariant Mass distribution;Mass;"

nbins = 100
xlow = 110
xup = 160
histo_name="h1"
h1 = ROOT.TH1F(histo_name, histo_title, nbins, xlow, xup)

# Dump the content of the tree into the TH1D
tree.Draw("x >> h1")

h1.GetXaxis().SetTitle("x")
h1.GetYaxis().SetTitle("Counts")
# Draw the histogram
c1 = ROOT.TCanvas("c1", "Histogram", 800, 600)
h1.Draw("E1 P")
c1.Draw()

Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


## 3. Model creation

We make now the model using the capabilities of TF1 using the NSUM operator (normalized sum of functions). 

We assume a Gaussian distribution for the signal and a double  exponential distribution for the background as following: 

$$ P(x | \mu , \nu ) = n_{sig} \times G(x | M , \sigma) + n_{bkg} \times E(x|a_1,a_2)$$

where $G (x | M , \sigma)$ is the Gaussian distribution for the signal and $E(x|a_1,a_2)$ is the exponential distribution describing the background. 

$$E(x|a_1,a_2) = \frac{ e^{( - a1 * x/100 - a2 * (x/100)^2 )}}{\int e^{-(.....)} dx }$$ 

In [4]:
# First create the signal, i.e a gaussian distribution: 'fsign'
fsign = ROOT.TF1("fsign","[Constant]*exp(-0.5*((x-[Mean])/[Sigma])**2)")

# Then the background distribution: the exponential
fbkg = ROOT.TF1("fbkg","[Constant]*exp(-([a1]*x)/100.-[a2]*(x/100)*(x/100))")

# And use NSUM to add them both in between 110 and 160: 'fmodel'
#fmodel = ROOT.TF1("fmodel","fsign + fbkg",110,160)
fmodel = ROOT.TF1("fmodel","NSUM(fsign,fbkg)",110,160)
#fmodel = ROOT.NSUM("fmodel",fsign,fbkg,110,160)

## 4. Fit the data 

We do first a fit to the histogram using the background only function to get reasonable values for the parameters

In [5]:
# A good starting guess for the Constant, i.e, number of background events, is the
# events available 
fbkg.SetParameter(0,30000) # 0 és l'ínedx del paràmetre Constant

# Set the other parameters
fbkg.SetParameter("a1",8) # 1 és l'índex del paràmetre a1
fbkg.SetParameter("a2",2) # 2 és l'índex del paràmetre a2

# And the range 
fbkg.SetParLimits(1,0,20)
fbkg.SetParLimits(2,0,5)

# Fit the data (h1) to the background function (using "L" options
h1.Fit("fbkg","L")

# Draw it
c2 = ROOT.TCanvas("c2", "Fit to histogram with background function", 800, 600)
h1.Draw("E")

legend = ROOT.TLegend(0.4, 0.75, 0.72, 0.85)
# Set legend font to 72 and the text size to 0.04
legend.SetTextFont(72)
legend.SetTextSize(0.04)

legend.AddEntry("fbkg", "Background fit", "l")
legend.Draw()

c2.Draw()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      67.6501
Chi2                      =        135.3
NDf                       =           97
Edm                       =  1.71012e-06
NCalls                    =          295
Constant                  =      23412.1   +/-   1272.44     
a1                        =      3.29054   +/-   0.0420855    	 (limited)
a2                        =   4.0323e-10   +/-   0.0304813    	 (limited)


### Full data fit using the created model: Higgs signal + background

We first set initial parameter in the function

In [6]:
# Print the fmodel to see the name of the parameters
fbkg.Print()

Formula based function:     fbkg 
                 fbkg : [Constant]*exp(-([a1]*x)/100.-[a2]*(x/100)*(x/100)) Ndim= 1, Npar= 3, Number= 0 
 Formula expression: 
	[Constant]*exp(-([a1]*x)/100.-[a2]*(x/100)*(x/100)) 


And set the parameters

In [7]:
# Parameters for the signal and sum part of the model

# [NSig, NBkg, Mean, Sigma, a1, a2]
fmodel.SetParameters(500, 30000, 125.0, 1.5, 8.0, 2.0)

# Set limits for some parameters
fmodel.SetParLimits(0, 0, 10000)
fmodel.SetParLimits(2, 120, 130)
fmodel.SetParLimits(3, 0.5, 3.0)

# Set the parameters a1 and a2 using the fitted values of fbkg
fmodel.SetParameter("a1",fbkg.GetParameter("a1"))
fmodel.SetParameter("a2",fbkg.GetParameter("a2"))

Now we fit the histogram. We perform a Binned likelihood fit (option L)

In [8]:
res = h1.Fit(fmodel,"L S")

# And draw  the model and the data
c3 = ROOT.TCanvas("c3", "Fit to histogram with full model", 800, 600)
h1.Draw("E")
fmodel.Draw("same")

legend = ROOT.TLegend(0.4, 0.75, 0.72, 0.85)
# Set legend font to 72 and the text size to 0.04
legend.SetTextFont(72)
legend.SetTextSize(0.04)

legend.AddEntry("fbkg", "Background fit", "l")
legend.AddEntry("fmodel", "Model fit", "l")
legend.Draw()

c3.Draw()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      64.9937
Chi2                      =      129.987
NDf                       =           94
Edm                       =  6.97561e-07
NCalls                    =          213
Coeff0                    =      59.3408   +/-   34.1234      	 (limited)
Coeff1                    =      15325.9   +/-   93.2885     
Mean                      =      124.472   +/-   0.68469      	 (limited)
Sigma                     =     0.864344   +/-   0.501308     	 (limited)
a1                        =      3.28212   +/-   0.042696    
a2                        = -1.09916e-06   +/-   4.2224e-05  


The number of signal/background events are equal to the Coefficients in the Normalized sum function divided by the bin width

In [9]:
bw = h1.GetBinWidth(1)
print("Number of Higgs  events = {0:5.0f} +/- {1:3.0f}".format(res.Parameter(0)/bw,res.ParError(0)/bw))
print("Number of Backg. events = {0:.0f} +/- {1:.0f}".format(res.Parameter(1)/bw,res.ParError(1)/bw))

Number of Higgs  events =   119 +/-  68
Number of Backg. events = 30652 +/- 187


## 5. Compute the Significance

For computing the significance we look at the Delta of the Likelihood obtained by fitting fixing the number of signal events to be zero and the full fit. 
The significance is equal to $\sqrt {\Delta logL}$.

We do then a background only fit 

In [10]:
# create a model with the number of signal events is zero, so how the model is
# behaving when there is no signal
fmodel2 = ROOT.TF1(fmodel) 
fmodel2.FixParameter(0,0)
fmodel2.FixParameter(2, res.Parameter(2))
fmodel2.FixParameter(3, res.Parameter(3))
fmodel2.SetLineColor(ROOT.kBlue)

In [11]:
# Do the fit and draw
res2 = h1.Fit(fmodel2,"L S +")
legend = ROOT.TLegend(0.4, 0.75, 0.72, 0.85)
# Set legend font to 72 and the text size to 0.04
legend.SetTextFont(72)
legend.SetTextSize(0.04)

legend.AddEntry("fmodel2", "Model2 fit", "l")
legend.AddEntry("fmodel", "Model fit", "l")
legend.Draw()

ROOT.gPad.Draw()

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      54.4763
Chi2                      =      108.953
NDf                       =           97
Edm                       =  2.63935e-06
NCalls                    =           73
Coeff0                    =            0                      	 (fixed)
Coeff1                    =      15385.2   +/-   87.7079     
Mean                      =      124.472                      	 (fixed)
Sigma                     =     0.864344                      	 (fixed)
a1                        =      7.65561   +/-   0.84228     
a2                        =      -1.6498   +/-   0.31785     


And we evaluate how it change the likelihood in absence of signal

In [12]:
print("Significance is = {0:.2f}".format(np.sqrt( np.abs(res2.MinFcnValue() - res.MinFcnValue() ))))
print("Model 1, signal + background", res.MinFcnValue())
print("Model 2, background", res2.MinFcnValue())

Significance is = 3.24
Model 1, signal + background 64.99370324177413
Model 2, background 54.47629791266442


In [ ]:
# Crec que el valor del model 2 hauria de ser més gran que el del model 1, ja que el model 1 té més paràmetres i per tant hauria d'ajustar millor.
# Tal vegada és per com he fixat els paràmetres.